<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Solution: When you have few labels

**[When you have few labels](https://learning.nextia-ai.com/courses/text-routing/m02/when-you-have-few-labels/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 2, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** choose which messages a person labels first (active learning), to check labels that an LLM wrote before you train on them, and to show with evidence whether more labels or better labels help more.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the LLM's labels were recorded once. |
| **You should know** | Wrong labels and what they cost, from [Find noisy labels](https://learning.nextia-ai.com/courses/text-routing/m02/find-noisy-labels/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m02/when-you-have-few-labels/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M02-L04-when-you-have-few-labels/when-you-have-few-labels-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Start with 70 labels

Imagine that Larkfield has no labels yet. Chloé labels 10 messages of each route: 70 messages. The other 5,930 training messages wait in a **pool** without labels.

This inbox already has a reviewed label for every training message. When the loop "asks Chloé", it reads that label. In real work, a person labels each picked message.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, FeatureUnion

def tfidf_model():
    """TF-IDF of words and letters with logistic regression: the course's model."""
    return make_pipeline(
        FeatureUnion([("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
                      ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True))]),
        LogisticRegression(C=10, max_iter=3000))

rng = np.random.default_rng(36)
start = np.concatenate([rng.choice(np.flatnonzero(train["route"].to_numpy() == r), 10, replace=False) for r in rt.ROUTES])
model = tfidf_model().fit(train["text"].iloc[start], train["route"].iloc[start])
print(len(start), "labels: test accuracy", (model.predict(test["text"]) == test["route"]).mean())

> **Check.** You should see `70 labels: test accuracy 0.5235`.

## 2. Which messages should Chloé label next?

The model gives each message in the pool a probability for each route. Its **confidence** is the highest of the seven. A message with confidence 0.95 teaches the model little: it already knows. A message with confidence 0.20 is one the model cannot place.

> **Predict.** The 100 least confident messages: which routes do they come from? The same mix as a random 100?

In [ ]:
pool = np.setdiff1d(np.arange(len(train)), start)
confidence = model.predict_proba(train["text"].iloc[pool]).max(axis=1)
least = pool[np.argsort(confidence)[:100]]
random_100 = np.random.default_rng(37).choice(pool, 100, replace=False)
print("median confidence in the pool:", round(float(np.median(confidence)), 3))
print("highest confidence among the 100 least confident:", round(float(np.sort(confidence)[99]), 3))
pd.DataFrame({"least confident": train["route"].iloc[least].value_counts(),
              "random": train["route"].iloc[random_100].value_counts()}).reindex(rt.ROUTES)

> **Check.** You should see a median of 0.379 and 0.203. The least confident 100 have 28 `returns_refunds` and only 5 `orders`; the random 100 have 18 `orders`.

## 3. The loop: random against least confident

Each round trains the model, scores the pool, and adds 100 labels: either at random, or the 100 the model is least sure about. Ten rounds take the labels from 70 to 1,070. The cell trains 22 models and takes about a minute.

In [ ]:
def active_learning(strategy, last=1070):
    labelled = list(start)
    pick_random = np.random.default_rng(37)
    curve = []
    while True:
        model = tfidf_model().fit(train["text"].iloc[labelled], train["route"].iloc[labelled])
        curve.append({"strategy": strategy, "labels": len(labelled),
                      "accuracy": round(float((model.predict(test["text"]) == test["route"]).mean()), 4)})
        if len(labelled) >= last:
            return curve
        pool = np.setdiff1d(np.arange(len(train)), labelled)
        if strategy == "random":
            add = pick_random.choice(pool, 100, replace=False)
        else:
            confidence = model.predict_proba(train["text"].iloc[pool]).max(axis=1)
            add = pool[np.argsort(confidence)[:100]]
        labelled.extend(add.tolist())

curves = pd.DataFrame(active_learning("random") + active_learning("least confident"))
table = curves.pivot(index="labels", columns="strategy", values="accuracy")
table

> **Check.** At 370 labels you should see 0.8475 (random) and 0.855 (least confident); at 1,070 labels 0.907 and 0.9375.

In [ ]:
import matplotlib.pyplot as plt
ax = table.plot(marker="o", figsize=(7, 4))
ax.set_xlabel("labelled messages"); ax.set_ylabel("test accuracy"); ax.grid(alpha=0.3)
plt.show()

## 4. Your turn: how many labels does active learning save?

> **Your turn.** Least confident labelling passes 0.90 accuracy with 670 labels. Find the smallest number of labels in `table` where **random** labelling reaches at least 0.90, and put it in `random_needs`.

In [ ]:
random_needs = int(table.index[table["random"] >= 0.90].min())
print(random_needs)

In [ ]:
expect('random_needs', random_needs, 970)

Random labelling needs 970 labels to pass 0.90; least confident labelling needed 670. Here, that is 300 labels that Chloé does not have to write for the same accuracy.

## 5. Labels written by an LLM

A small hosted LLM labelled 5,000 training messages, 20 per call, for US$0.081 in all. Each answer has a route and a confidence that the model **wrote**. The answers were recorded once; the file is `data/recordings/llm-labels.jsonl`.

> **Predict.** What share of the LLM's labels agree with the reviewed labels?

In [ ]:
llm = pd.read_json("data/recordings/llm-labels.jsonl", lines=True)
llm = llm.merge(train[["message_id", "route"]].rename(columns={"route": "people"}), on="message_id")
llm["agree"] = llm["route"] == llm["people"]
print(f"{len(llm)} labels, agree with people: {llm['agree'].mean():.1%}, cost US${llm['usd'].sum():.3f}")
bands = pd.cut(llm["confidence"], [0, 0.8, 0.9, 0.95, 1.01], right=False,
               labels=["below 0.80", "0.80 to 0.90", "0.90 to 0.95", "0.95 or more"])
llm.groupby(bands, observed=True)["agree"].agg(["size", "mean"]).round(3)

> **Check.** You should see `5000 labels, agree with people: 91.4%, cost US$0.081`. Below a written confidence of 0.80, only 66.4% of 336 labels agree; at 0.95 or more, 95.0% of 3,995.

The written confidence helps: low confidence means more errors. So a person should review the least confident labels first, as in active learning.

In [ ]:
ordered = llm.sort_values("confidence", kind="stable").reset_index(drop=True)
for n in (0, 250, 500, 1000, 2000):
    errors_left = int((~ordered["agree"].iloc[n:]).sum())
    print(f"review the {n:>4} least confident: {errors_left} errors left, review cost US${n * 1.5:,.0f}")

> **Check.** You should see 429 errors with no review, 198 after 1,000 reviews and 98 after 2,000. A review costs US$1.50 (the course's cost table, an assumption).

## 6. Your turn: the errors the LLM is sure about

> **Your turn.** Count the LLM labels that are wrong (`agree` is False) **and** have a written confidence of 0.95 or more. Put the count in `sure_but_wrong`.

In [ ]:
sure_but_wrong = int((~llm["agree"] & (llm["confidence"] >= 0.95)).sum())
print(sure_but_wrong)

In [ ]:
expect('sure_but_wrong', sure_but_wrong, 198)

A review by confidence never sees these 198 labels. Check a **random** sample of the LLM's labels too, and measure their agreement before you train on them.

## 7. More labels or better labels?

The last cell answers the module's question with two models. One learns from all 6,000 old, noisy labels (see the last lesson). The other learns from only 2,000 reviewed labels.

In [ ]:
old = pd.read_csv("data/old_queue.csv")
tr = train.merge(old, on="message_id")
sub = tr.sample(2000, random_state=36)
more = tfidf_model().fit(tr["text"], tr["old_queue"]).predict(test["text"])
better = tfidf_model().fit(sub["text"], sub["route"]).predict(test["text"])
rt.show([rt.score("6,000 old labels", test, more), rt.score("2,000 reviewed labels", test, better)])

> **Check.** You should see accuracy 0.934 and US$655.0 per 1,000 messages for the 2,000 reviewed labels, and 0.913 and US$1,206.0 for the 6,000 old labels.

Here, a third of the labels, if they are right, beat all the noisy labels. The lesson page shows the whole curve.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m02/when-you-have-few-labels/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Keyword rules and regular expressions](https://learning.nextia-ai.com/courses/text-routing/m03/keyword-rules-and-regular-expressions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m02/when-you-have-few-labels/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.